## Notebook setup

#### Import packages

In [ ]:
!pip install h3
!pip install esda
!pip install splot
!pip install -U gdown
!pip install duckdb
!pip install folium

from datetime import timedelta
import duckdb
from esda.moran import Moran
import folium
import geopandas as gpd
import gdown
import h3
from libpysal.weights import Queen, KNN
import math
import matplotlib
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'monospace'
import numpy as np
import os
import pandas as pd
import pickle
import random
from scipy.stats import pearsonr
import seaborn as sns
from shapely.geometry import Polygon, Point, mapping
from shapely import wkt
from splot.esda import moran_scatterplot
import time
from tqdm import tqdm
tqdm.pandas()
import urllib.request
import zipfile


#### Plotting functions
These will help us to visualize and explore the data --- press the play button below to run the cell but **feel free to ignore** the content.

In [ ]:
def choropleth_folium(gdf, key_col, val_col, m = None, fill_cmap = 'YlGnBu'):
    gdf = gdf.to_crs(4326)
    if m == None:
      m = folium.Map(width = 850, height = 500, location=[gdf.geometry.centroid.y.mean(), gdf.geometry.centroid.x.mean()], zoom_start=10,tiles="Cartodb Positron")
    folium.Choropleth(
        geo_data=gdf.to_json(),
        data=gdf,
        columns=[key_col, val_col],
        key_on=f'feature.properties.{key_col}',
        fill_color=fill_cmap,
        fill_opacity=0.7,
        line_opacity=0.2,
        legend_name=val_col
    ).add_to(m)
    return m

def plot_data_folium(gdf, m = None, shape = 'circle', color='blue', hw = True):
    # Ensure the GeoDataFrame is in WGS84 (lat/lon) for Folium
    if gdf.crs != "EPSG:4326":
        gdf = gdf.to_crs(epsg=4326)

    # Get map center
    map_center = [gdf.geometry.y.mean(), gdf.geometry.x.mean()]

    if not m:
    # Create base map
        m = folium.Map(width=850, height=500, location=map_center, zoom_start=11,tiles="Cartodb Positron")

    # Add points as simple dots (CircleMarker is lightweight and looks like a dot)
    for _, row in gdf.iterrows():
        if hw:
            if row['activity_type'] == 'Home':
                c = 'green'
                s = 2
                o = 1
                # icon=folium.DivIcon(html=f"""<div style="font-size: 12pt; color: black;">Home</div>""")
            elif row['activity_type'] == 'Work':
                c = 'red'
                s = 2
                o = 1
                # icon=folium.DivIcon(html=f"""<div style="font-size: 12pt; color: black;">Work</div>""")
            else:
                c = color
                s = .5
                o = .5
                # icon=None
        else:
            c = color
            s = 2
            o = 1
        tooltip_content = f"""
                            <b>Start time:</b> {row['start_time']}<br>
                            <b>End time:</b> {row['end_time']}<br>
                            <b>Duration:</b> {round(row['duration'],2)} min<br>
                            <b>Activity type:</b> {row['activity_type']}<br>
                            """
        if shape == 'square':
            folium.RegularPolygonMarker(
                location=[row.geometry.y, row.geometry.x],
                radius=5,  # Adjust dot size
                fill_color=color,
                color=c,
                number_of_sides = 4,
                fill_opacity=o,
                weight=s,
                tooltip=folium.Tooltip(tooltip_content, sticky=True)
                # icon=icon
            ).add_to(m)
        else:
            folium.CircleMarker(
                location=[row.geometry.y, row.geometry.x],
                radius=5,  # Adjust dot size
                fill_color=color,
                color=c,
                number_of_sides = 4,
                fill_opacity=o,
                weight=s,
                tooltip=folium.Tooltip(tooltip_content, sticky=True)
                # icon=icon
            ).add_to(m)
    return(m)

def get_endpoint(x1, y1, x3, y3, k):
    # Compute direction vector
    dx = x3 - x1
    dy = y3 - y1
    length = math.hypot(dx, dy)

    if length == 0:
        raise ValueError("Direction vector cannot be zero-length.")

    # Normalize and scale by k
    dx_scaled = (dx / length) * k
    dy_scaled = (dy / length) * k

    # Compute endpoint
    x2 = x1 + dx_scaled
    y2 = y1 + dy_scaled

    return x2, y2

def degrees_to_meters(lat_deg, lon_deg, latitude):
    # Approximate conversion
    meters_lat = lat_deg * 111320
    meters_lon = lon_deg * 111320 * math.cos(math.radians(latitude))
    return meters_lat, meters_lon

def plot_rad(user_df, color, m=None, rad=None):
    if not rad:
        rad = radius_gyration(user_df)

    if user_df.crs != "EPSG:4326":
        user_df = user_df.to_crs(epsg=4326)

    # Get map center
    map_center = [user_df.geometry.y.mean(), user_df.geometry.x.mean()]

    if not m:
    # Create base map
        m = folium.Map(width=850,height=500, location=map_center, zoom_start=11,tiles="Cartodb Positron")

    if rad>0:
        folium.CircleMarker(location=map_center,
                        radius=5, fill=True, # Adjust dot size
                        fill_color = color, color=color, fill_opacity=1).add_to(m)

        folium.Circle(location=map_center,
                        radius=rad, fill=True,# Adjust dot size
                        fill_color = color, color=color, fill_opacity=.2).add_to(m)
    return(m)

def plot_overlaps(h3_code='88195da49bfffff',res=8,day=25,hour=19):
    dem_tab = f"read_csv('{input_path}msoa_income.csv', header = 1)"
    Q = 5
    dem_info = f'CEILING((PERCENT_RANK() OVER (ORDER BY inc))*{Q})'

    cmap = matplotlib.colormaps.get_cmap("Spectral")

    color_dict= {1:tuple(round(i*255) for i in cmap(0.00)[:3]),
                 2:tuple(round(i*255) for i in cmap(0.25)[:3]),
                 3:tuple(round(i*255) for i in cmap(0.50)[:3]),
                 4:tuple(round(i*255) for i in cmap(0.75)[:3]),
                 5:tuple(round(i*255) for i in cmap(0.99)[:3])}

    labels = ['No income available','Low income','Med-low','Med','Med-high','High income']
    hex_poly = Polygon([(pt[1],pt[0]) for pt in h3.cell_to_boundary(h3_code)])
    center = np.mean(np.asarray(h3.cell_to_boundary(h3_code)),axis=0)
    # Create a Folium map centered on the polygon
    polymap = folium.Map(height = 500, width = 850, location=center, zoom_start=14,tiles="Cartodb Positron")

    # Add the polygon to the map using GeoJson
    folium.GeoJson(hex_poly, name="shapely-polygon",color='grey').add_to(polymap)

    q = f"""
        WITH
        home_tab AS (
            SELECT DISTINCT userid, a.loc_msoa as home_msoa, {dem_info} as dem_info
            FROM {df} a
            FULL OUTER JOIN {dem_tab} b
            ON a.loc_msoa = b.msoa21cd
            WHERE activity_type = 'Home'
            AND o_lat IS NOT NULL and o_lon IS NOT NULL
        )

        SELECT * FROM {df} d
            FULL OUTER JOIN home_tab h ON d.userid = h.userid
            WHERE o_h{res} = '{h3_code}'
            AND EXTRACT(DAY FROM start_time) = {day}
            AND EXTRACT(HOUR FROM start_time) = {hour}
        """
    users_present = con.execute(q).df()
    users_present['geometry'] = users_present.geometry.apply(wkt.loads)
    users_present= gpd.GeoDataFrame(users_present).set_crs(27700,allow_override=True)
    # Build the HTML legend dynamically
    legend_items = f"""
        <div style="display: flex; align-items: center; margin-top: 5px;">
            <div style="width: 12px; height: 12px; background-color: lightgrey; border-radius: 50%; margin-right: 8px;"></div>
            No income available
        </div>
        """
    for i in range(1, 6):
        legend_items += f"""
        <div style="display: flex; align-items: center; margin-top: 5px;">
            <div style="width: 12px; height: 12px; background-color: rgb{color_dict[i]}; border-radius: 50%; margin-right: 8px;"></div>
            {labels[i]}
        </div>
        """

    legend_html = f"""
    <div style="
        position: fixed;
        top: 50px; left: 50px; width: 180px; height: auto;
        background-color: white;
        border:2px solid grey;
        z-index:9999;
        font-size:14px;
        padding: 10px;
    ">
    <b>Income quintile</b><br>
    {legend_items}
    </div>
    """
    # Define title HTML
    title_html = f"""
    <h3 align="center" style="font-size:20px"><b>{h3_code}: November {day}, {hour}:00</b></h3>
    """

    # Add title to map
    polymap.get_root().html.add_child(folium.Element(title_html))

    polymap.get_root().html.add_child(folium.Element(legend_html))

    for i in range(len(users_present)):
        if users_present[i:i+1].dem_info.values[0] == users_present[i:i+1].dem_info.values[0]:
            cmap_loc = (users_present[i:i+1].dem_info.values[0]-1)/(np.max(users_present.dem_info.dropna())-1)
            color = f'rgb{str(tuple(float(i*255) for i in cmap(cmap_loc)[:3]))}'
        else:
            color='grey'
        polymap = plot_data_folium(users_present[i:i+1],polymap,color=color,hw=False)
    return polymap


 #### Load in data

In [ ]:
url = "https://drive.google.com/drive/folders/1dp-bXb-qHb5Esq354l0j00DRD_isXVno"

paths = gdown.download_folder(
    url=url,
    output="./data",
    quiet=False,
    use_cookies=False,
)

# Change to the filepath where the data is stored
input_path = 'data/'

# Load in census geometry shapefiles
bfcs_london = gpd.read_file(f'{input_path}bfcs_london.geojson') # Lower super output area
bfcs_msoa = bfcs_london.dissolve(by='msoa21cd').reset_index() # Middle super output area
bfcs_msoa['lad'] = bfcs_msoa.label.str[-27:-18] # Local authority district
bfcs_lad = bfcs_msoa.dissolve(by='lad').reset_index()
msoas_london = set(bfcs_london.label.str[-18:-9])

# Load in mobility data
travel_data = gpd.read_file(f"{input_path}df_samp.geojson")
df = f"read_csv('{input_path}df_samp.csv',header=1)"

## Initial data exploration

Our primary dataset is a synthesized mobility dataset derived from the open data [here](https://github.com/c-zhong-ucl-ac-uk/Anonymised-human-location-data-for-urban-mobility-research).

 This dataset represents mobile phone traces from 5000 users in London, which have already been processed into activities via the stop detection methods and home location methods touched upon in Part I of this tutorial. The original data processing, including stop detection and home location detection, is described [here](https://www.researchgate.net/profile/Nilufer-Sari-Aslam/publication/383185185_Anonymised_Human_Location_Data_for_Urban_Mobility_Research/links/66c09eb1145f4d355360bba2/Anonymised-Human-Location-Data-for-Urban-Mobility-Research.pdf).


First, look at the columns that we have.

- **userid**: unique hash code representing a single user in our dataset
- **start_time**: the time at which the given activity started (note that it is rounded to the nearest 15 minutes)
- **end_time**: the time at which the given activity ended
- **duration**: exact duration of the given activity, in seconds
- **loc_msoa**: the Middle Super Output Area (MSOA) in which the activity took place. in this synthesized dataset, more specific location information is randomized within this MSOA.
- **activity_type**: labelled activity type. Home and work locations are estimated as the location where the user spends the most daytime and nighttime hours, respectively; other activity types are estimated using the nearest point of interest (POI) to the activity's precise location.
- **o_lat, o_lon**: UTM coordinates of the location  (in this synthesized dataset, randomized within MSOA). CRS is EPSG:2770.
- **o_h7, o_h8, o_h9**: h3 geohash levels 7,8, and 9 containing the coordinates (**o_lat**, **o_lon**)
- **home_activity**: 'Y' if activity_type == 'Home', 'N' otherwise
- **geometry**: Point object associated with the coordinates (**o_lat**, **o_lon**)

You can see the first 5 rows of the data below.

In [ ]:
travel_data.head()

**Note**: You may notice that our latitude and longitude columns, o_lat and o_lon have already been converted into x-y coordinates (Northing, Easting) --- this makes it easier for us to calculate distances.

Now, let's look at how much data we have for each user.

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,5))

# Histogram of activities observed per user.
activities_peruser = travel_data.groupby('userid').size()
days_peruser = travel_data.groupby('userid').agg({'start_time':lambda x: x.dt.date.nunique()}).start_time
activities_perday = activities_peruser/days_peruser

ax[0].hist(activities_perday,bins=50,edgecolor='black')
ax[0].set_ylabel('Number of users')
ax[0].set_xlabel('Number of activities observed')

# Histogram of unique days observed per user.
ax[1].hist(travel_data.groupby('userid').agg({'start_time':lambda x: x.dt.date.nunique()}),bins=range(1,32),edgecolor='black')
ax[1].set_xticks(ticks = [i+.5 for i in range(1,31) if i%2==0], labels = range(2,32,2))
ax[1].set_ylabel('Number of users')
ax[1].set_xlabel('Number of unique days observed')
fig.suptitle(f'Descriptive data for {len(set(travel_data.userid))} total users')


Okay great --- we can see that for most users, we observe between 1 and 5 activities per day. For around 20% of our users, we only observe them on a single day; that said, we see around 10% of our users every single day in November.

What does this data actually look like? **Let's look at some sample users --- how do they move around the city of London?**

I have picked two sample users for us who behave differently from one another --- but if you want to randomly select two new users, you can do so by uncommenting the second group of lines in the code chunk below.

We'll plot their observed activities on a map of London. **Each data point is an activity, or a cluster of points representing a stop at a single location.**  Estimated home locations are marked in **green** and work locations marked in **red**.

In [ ]:
sample_user = '1E548A4F633DAF3F06E1FFE45A8B6EE1'
sample_user2 = 'E43D63713E0182CF77856EF6150B84BF'

# # UNCOMMENT THE LINES BELOW TO SELECT NEW, RANDOM USERS
# # (you may have to zoom out using the controls in the top left corner of the map to see one or both users)
# sample_user = random.sample(list((travel_data.userid)),k=1)[0]
# sample_user2 = random.sample(list((travel_data.userid)),k=1)[0]

sample_user_data = travel_data[travel_data.userid==sample_user]
sample_user_data2 = travel_data[travel_data.userid==sample_user2]

m = plot_data_folium(sample_user_data, color='cornflowerblue') # Plot the first user's data. The plotting function is defined above.
m = plot_data_folium(sample_user_data2, m, shape = 'square', color='orange') # Plot the second user's data on the same map m,
                                                                             # with a different shape marker and a different color.
m

Sample user 1 and sample user 2 have very different activity spaces --- user 2 lives and works in the far west of London, whereas user 1 lives in northeast London and their activities are concentrated near central London. That said, they do have some overlap, potentially encountering one another near Cricklewood.

**What are some ways that we can quantify our sample users' mobility patterns?** In the next section, we will calculate mobility statistics which can be used to quantify and compare mobility patterns.

## Calculate radius of gyration

We are going to calculate 3 mobility characteristics, which are explored in [this paper](https://journals.sagepub.com/doi/abs/10.1177/23998083241234137):
1. **Radius of gyration**: Characteristic distance of a user's daily travels
2. **Self-containment**: Tendency of a user to stay close to home
3. **Social interaction potential**: Exposure of a user to other people throughout their daily movements

We're going to start with radius of gyration, and then repeat the same process for our other two metrics.




### Radius of gyration
**The radius of gyration represents the size of an individual's activity space**: how far they travel to reach urban services as they move throughout their daily lives.

It is calculated using the following formula:  

\begin{align}
R_g &= \sqrt{\frac{\sum_{i=1}^{N_{\text{stay}}} \left[ \left(P_i^x - \bar{P}^x \right)^2 + \left(P_i^y - \bar{P}^y \right)^2 \right]}{N_{\text{stay}}}}
\end{align}

where $(\bar{P}^x, \bar{P}^y)$ are the average x, y points of the user's activities respectively. Let's calculate the radius of gyration of our sample users with a python function.

In [ ]:
def radius_gyration(data, latcol = 'o_lat', loncol = 'o_lon'):
    # Find average xy-coordinate
    xmean = np.mean(data[loncol])
    ymean = np.mean(data[latcol])
    nstay = len(data)

    # Calculate radius of gyration
    rad_gyr = np.sqrt(np.sum((data[latcol]-ymean)**2 + (data[loncol]-xmean)**2)/nstay)
    return rad_gyr

start = time.time()
print(f'Radius of gyration, user 1: {np.round(radius_gyration(sample_user_data),3)}')
print(f'Radius of gyration, user 2: {np.round(radius_gyration(sample_user_data2),3)}')


**Radius of gyration is around 3x larger for sample user 1 than for sample user 2**, as would be expected from our map of their trajectories --- sample user 1 generally traverses a wider range of the city.

This is visualized in the plots below, where the dark point represents the centre points of each users' activities, and the circle extending outward from it has radius $r_g$ = radius of gyration.



In [ ]:
sample_user = '1E548A4F633DAF3F06E1FFE45A8B6EE1'
sample_user2 = 'E43D63713E0182CF77856EF6150B84BF'

## AGAIN, YOU CAN UNCOMMENT THE LINES BELOW TO SELECT NEW, RANDOM USERS
## (you may have to zoom out using the controls in the top left corner of the map to see one or both users)
# sample_user = random.sample(list((travel_data.userid)),k=1)[0]
# sample_user2 = random.sample(list((travel_data.userid)),k=1)[0]

sample_user_data = travel_data[travel_data.userid==sample_user]
sample_user_data2 = travel_data[travel_data.userid==sample_user2]

m = plot_data_folium(sample_user_data, color='cornflowerblue') # Plot the first user's data. The plotting function is defined above.
m = plot_data_folium(sample_user_data2, m, shape = 'square', color='orange') # Plot the second user's data on the same map m,
m = plot_rad(sample_user_data,m=m,color='darkblue')
m = plot_rad(sample_user_data2,m=m,color='chocolate')

m

We might be curious --- what are some underlying drivers of differences in radius of gyration? Are people with certain characteristics --- for example, those who live nearer to central London, those who have higher income --- more likely to explore larger geographical extents of the city? In order to begin to answer this question, we must calculate radius of gyration for all the users in our dataset.

#### Expanding to the full dataset

We use the same Python function to calculate radius of gyration for our full sample of users below.


In [ ]:
# Apply the radius of gyration function to the full dataset.
rad_gyr_df = travel_data.groupby('userid').progress_apply(radius_gyration).reset_index(name='rad_gyr')

# Add users' home information.
homes_df = travel_data[travel_data.home_activity=='Y'].groupby('userid').agg(home_lat = ('o_lat','mean'),
                                                                                          home_lon = ('o_lon','mean'),
                                                                                          home_msoa = ('loc_msoa','min')).reset_index()
rad_gyr_df = rad_gyr_df.merge(homes_df,on='userid')
rad_gyr_df.head()


#### SQL implementation
**Note**: It probably took somewhere around 5 seconds to run this calculation for the full set of 5,000 users. This could start to add up for much larger set of users --- say, 100,000 users --- or longer timeframes --- say, a full year of data. In order to run fast, efficient calculations on a larger dataset, we can use SQL. See hidden cells below for a SQL implementation of radius of gyration using the Python package DuckDB.


In [ ]:
con = duckdb.connect()
con.install_extension("spatial")
con.load_extension("spatial")

df_path = f"read_csv('{input_path}df_samp.csv',header=1)"

find_radius_of_gyration = f"""
            WITH user_avgs AS (
                    SELECT userid,
                           AVG(o_lon) AS avg_lon,
                           AVG(o_lat) AS avg_lat,
                           COUNT(*) AS total_points
                    FROM {df_path}
                    GROUP BY userid
                ),

                home_tab AS (
                    SELECT DISTINCT userid, lsoa21cd as home_lsoa, o_lat as home_lat, o_lon as home_lon
                    FROM {df_path} a
                    INNER JOIN read_csv('{input_path}oa_lsoa.csv', header=1) b
                    ON a.o_oa = b.oa21cd
                    WHERE home_activity = 'Y'
                )

                SELECT a.userid,
                       h.home_lsoa, h.home_lat, h.home_lon,
                       SQRT(SUM(POWER(o_lon - avg_lon, 2) + POWER(o_lat - avg_lat, 2)) / ANY_VALUE(a.total_points)) AS rad_gyr
                FROM {df_path} t
                JOIN user_avgs a ON t.userid = a.userid
                JOIN home_tab h ON t.userid = h.userid
                GROUP BY a.userid, h.home_lsoa, h.home_lon, h.home_lat
            """
start = time.time()
rad_gyr_df = con.execute(find_radius_of_gyration).df()
print(f'Calculation time: {time.time() - start}')

####Data exploration

Let's explore what the distribution of radius of gyration looks like across our sample users.

First, take a look at the table we've just created:
- **userid**: unique user id
- **home_lat**: latitude of user's home location
- **home_lon**: longitude of user's home location
- **home_lsoa**: Lower Super Output Area (LSOA) containing the point (**home_lat**,**home_lon**)
- **rad_gyr**: radius of gyration, in meters

As you can see in the histogram below, there is a peak around 4km. The radius of gyration of our sample users are marked in blue (user 1) and orange (user 2) --- most users' behavior lies somewhere between the two.


In [ ]:
plt.hist(rad_gyr_df.rad_gyr,bins=60,color='lightgrey',edgecolor='grey',alpha=.5)
plt.axvline(rad_gyr_df[rad_gyr_df.userid==sample_user].rad_gyr.values[0],
            color='darkblue',
            linewidth=1.5,
            label='Sample user 1')
plt.axvline(rad_gyr_df[rad_gyr_df.userid==sample_user2].rad_gyr.values[0],
            color='chocolate',
            linewidth=1.5,
            label='Sample user 2')
plt.axvline(rad_gyr_df.rad_gyr.mean(),
            color='black',
            linewidth=2,
            linestyle='-.',
            label=f'Mean: {np.round(rad_gyr_df.rad_gyr.mean()/1000,2)}km')
plt.axvline(rad_gyr_df.rad_gyr.median(),
            color='black',
            linewidth=2,
            linestyle='dotted',
            label=f'Median: {np.round(rad_gyr_df.rad_gyr.median()/1000,2)}km')
plt.legend()
plt.title('Radius of gyration')

The median radius of gyration is indicated with a dotted line above. Our next mobility metric, **self-containment**, explores behaviour relative to this median.

####**Exercise** (optional)

Find one user with particularly high and one user with particularly low radius of gyration and plot their data as we did above. Repeat a few times to explore these extreme radius of gyration values. What types of users appear as outliers in our data? Do you think these users represent outliers in terms of lifestyle, or data anomalies of some kind?

In [ ]:
## First, try sorting the travel_data dataframe by "rad_gyr" column in order to identify high and low rg users.
## Then, paste their userids in the quotation marks to define sample_user_highrg and sample_user_lowrg.
## Uncomment the lines below to plot and explore the trajectories of high or low radius of gyration users.

# rad_gyr_df_sort = rad_gyr_df.sort_values('rad_gyr', ascending= False)

# sample_user_highrg = ''
# sample_user_lowrg = ''

# sample_user_data_highrg = travel_data[travel_data.userid==sample_user_highrg]
# sample_user_data_lowrg = travel_data[travel_data.userid==sample_user_lowrg]

# m = plot_data_folium(sample_user_data_highrg, color='cornflowerblue') # Plot the first user's data. The plotting function is defined above.
# m = plot_data_folium(sample_user_data_lowrg, m, shape = 'square', color='orange') # Plot the second user's data on the same map m,
# m = plot_rad(sample_user_data_highrg,m=m,color='darkblue')
# m = plot_rad(sample_user_data_lowrg,m=m,color='chocolate')

# m

#### Social and spatial patterns

**We have calculated average radius of gyration at the home LSOA level** across our full sample of several hundred thousand users and provided it in the tutorial files.

We also have information on median income and distance from the city center of each neighborhood.

The relevant columns are below:
- **lsoa21cd**: unique id of LSOA
- **rad_gyr**: radius of gyration
- **pctile_50**: median income within the LSOA --- an experimental statistic documented [here](https://www.ons.gov.uk/peoplepopulationandcommunity/personalandhouseholdfinances/incomeandwealth/datasets/adminbasedincomestatisticsdataforindividualsenglandandwales)
- **dist_from_center**: distance to the center of London, represented by the geographic center of the Greater London Area (531213.287 Easting 179649.909 Northing --- near [Waterloo Station](https://colab.research.google.com/drive/1JADtoZcHqLAV183yoUshPwoBImEhBrnq#scrollTo=DIfnoKcS2TB8&line=11&uniqifier=1))

Let's explore differences across space and socioeconomic groups!

In [ ]:
## Load in data
rad_gyr_alldata = pd.read_csv(f'{input_path}mobility_characteristics.csv')[['home_lsoa','rad_gyr']]
income = pd.read_csv(f'{input_path}lsoa_income.csv')
rad_gyr_alldata = rad_gyr_alldata.merge(income[['lsoa21cd','pctile_50']].rename(columns={'lsoa21cd':'home_lsoa'}),on='home_lsoa')

## Add geography
rad_gyr_alldata = bfcs_london.merge(rad_gyr_alldata.rename(columns={'home_lsoa':'lsoa21cd'}),on='lsoa21cd')

## We're going to use the geographic center of the GLA as our "city center" to help us think about spatial trends ---
## There are lots of ways to think about city centers, this isn't necessarily the best one.
london_center = rad_gyr_alldata.dissolve().centroid.values[0]
rad_gyr_alldata['dist_from_center'] = rad_gyr_alldata.geometry.apply(lambda x: x.centroid.distance(london_center))

**Does radius of gyration exhibit spatial patterns?**





Below, we have plotted $rad_{gyr}$ --- the average radius of gyration for residents of each LSOA.

In [ ]:
m = choropleth_folium(rad_gyr_alldata, 'lsoa21cd', 'rad_gyr')
m

There appears to be a broad spatial trend where **residents who live further from the center of London have a higher radius of gyration** (and thus tend to travel further from home). **Let's quantify this** apparent trend by calculating the Pearson correlation between distance to the city center and radius of gyration.

In [ ]:
plt.scatter(rad_gyr_alldata.dist_from_center, rad_gyr_alldata.rad_gyr,s=5)
plt.xlabel('Distance from city center')
plt.ylabel('Radius of gyration')

r = pearsonr(rad_gyr_alldata.dist_from_center, rad_gyr_alldata.rad_gyr)
print(f"Pearson correlation coefficient: {np.round(r[0],4)}")
print(f"p-value: {np.round(r[1],4)}")

The trend we saw in our map does seem to hold: residents who live further from the city center tend to have a higher radius of gyration, as evidenced by a moderate (and highly significant) Pearson's correlation coefficient of .54.

We may want to know, more generally: **is radius of gyration is clustered in space?** Let's calculate the [Moran's I statistic](https://pro.arcgis.com/en/pro-app/latest/tool-reference/spatial-statistics/h-how-spatial-autocorrelation-moran-s-i-spatial-st.htm) to evaluate how spatially correlated radius of gyration is.

In [ ]:
# Create a queen contiguity weights matrix
w = Queen.from_dataframe(rad_gyr_alldata)

# Alternatively: you could create a K-Nearest Neighbors contiguity matrix
# w = KNN.from_dataframe(gdf, k=5)

w.transform = 'r'  # Row-standardized weights
y = rad_gyr_alldata['rad_gyr'].values # Variable of interest here is radius of gyration
moran = Moran(y, w)

print("Moran's I:", moran.I)
print("p-value:", moran.p_sim)

# We can visualize this spatial autocorrelation with a moran scatterplot.
# Here, the x-axis represents radius of gyration (standard deviation-normalized)
# The y-axis is the average radius gyration of neighbors as defined by our contiguity matrix w (adjacent polygons)
fig, ax = moran_scatterplot(moran, aspect_equal=True)
plt.show()

We have a moderately strong Moran's I value of .45 and a highly significant p-value of .001 --- this indicates that **radius of gyration is clustered in space**. In other words, neighborhoods tend to have similar average radius of gyration as other neighborhoods geographically near to them.

This is illustrated in the Moran scatterplot above --- on the x axis we have normalized radius of gyration for a given neighborhood, and on the y axis we have average normalized radius of gyration of the adjacent neighborhoods as defined by our weights matrix w. Remember, we used a Queen weights matrix which means adjacency is defined as all neighborhoods which share any amount of border. **You can see in the scatter plot that a neighborhood's average radius of gyration is correlated with adjacent neighborhoods' average radii of gyration.**

## Repeat for self-containment, social interaction potential

### Self-containment

Self-containment describes **the propensity of individuals to stay close to home**. It is calculated as the percentage of non-home activities when are closer to home than the median radius of gyration in a given sample:

$$S_c^{non-home} = \frac{\sum_{i\in \text{non-residence activities}} \theta_i}{N_{\text{non-residence activities}}}$$

$$\theta_i = \begin{cases}1 \text{ if } \sqrt{(P_i^x - P_{home}^x)^2 + (P_i^y - P_{home}^y)^2} \leq Median(R_g)\\0 \text{ otherwise} \end{cases}$$

Let's calculate self-containment for our sample users with a simple python function.

In [ ]:
## First, we'll need to grab the median radius of gyration from our table.
rg_med = np.quantile(rad_gyr_df.rad_gyr,.5)

def self_containment(user_df):
    user_df = user_df.copy()
    if (len(user_df[user_df.activity_type=='Home'])>0) & (len(user_df[user_df.activity_type!='Home'])>0):
      home_pt = Point(*user_df[user_df.activity_type=='Home'][['o_lon','o_lat']].values[0])
      user_df['dist_from_home'] = user_df['geometry'].apply(lambda x: x.distance(home_pt))
      all_nonhome_activities = user_df[user_df.activity_type!='Home']
      close_nonhome_activities = all_nonhome_activities[all_nonhome_activities.dist_from_home<rg_med]
      return (len(close_nonhome_activities)/len(all_nonhome_activities))

start = time.time()
print(f'Self-containment, user 1: {np.round(self_containment(sample_user_data),3)*100}% of time spent < Median(R_g) from home.')
print(f'Self-containment, user 2: {np.round(self_containment(sample_user_data2),3)*100}% of time spent < Median(R_g) from home.')
print(f'Calculation time: {time.time() - start} seconds')

**Self-containment, again, is quite different for our two users.** It is over 3x larger for sample user 2 than for sample user 1, as would be expected from our map of their trajectories --- sample user 2 stays close to home a much larger proportion of the time.

This is visualized in the plots below, where the grey circles represent the extent of the median radius of gyration, centered at the users' home locations. User 1 (blue) takes frequent trips beyond the extent of the circle, while user 2 (orange) takes just a few trips outside the circle.

In [ ]:
home_pt = sample_user_data[sample_user_data.activity_type=='Home'].geometry.values[0]
sample_user_data.loc[:,'dist_from_home'] = sample_user_data['geometry'].apply(lambda x: x.distance(home_pt))
home_pt2 = sample_user_data2[sample_user_data2.activity_type=='Home'].geometry.values[0]
sample_user_data2.loc[:,'dist_from_home'] = sample_user_data2['geometry'].apply(lambda x: x.distance(home_pt2))

home_pt = sample_user_data[sample_user_data.activity_type=='Home'].to_crs(4326).geometry.values[0]
home_pt2 = sample_user_data2[sample_user_data2.activity_type=='Home'].to_crs(4326).geometry.values[0]

map_rg = plot_data_folium(sample_user_data,color='cornflowerblue')
map_rg = plot_data_folium(sample_user_data[sample_user_data.dist_from_home<rg_med],m=map_rg,color='darkblue')
folium.Circle(location = (home_pt.y,home_pt.x),radius=rg_med,color='lightgrey',fill_color='grey',alpha=.2).add_to(map_rg)

map_rg = plot_data_folium(sample_user_data2,map_rg,shape='square',color='orange')
map_rg = plot_data_folium(sample_user_data2[sample_user_data2.dist_from_home<rg_med],m=map_rg,shape='square',color='chocolate')
folium.Circle(location = (home_pt2.y,home_pt2.x),radius=rg_med,color='lightgrey',fill_color='grey',alpha=.2).add_to(map_rg)

map_rg


#### Expanding to the full dataset

Again, we can expand to our full dataset using our Python function.

In [ ]:
self_df = travel_data.groupby('userid').progress_apply(self_containment).reset_index(name='cont_nonhome')

self_df.head()

#### SQL implementation
**Note**: It probably took around 30 seconds to run this calculation for the full set of 5,000 users. Again, this could start to add up for much larger set of users --- say, 100,000 users --- or longer timeframes --- say, a full year of data. In order to run fast, efficient calculations on a larger dataset, we can use SQL. See hidden cells below for a SQL implementation of radius of gyration using the Python package DuckDB.


In [ ]:
self_cont = f"""
    WITH
    home_tab AS (
        SELECT DISTINCT userid, o_lon AS home_lon, o_lat AS home_lat, lsoa21cd as home_lsoa
        FROM {df_path} a
        INNER JOIN read_csv('{input_path}oa_lsoa.csv', header=1) b
        ON a.o_oa = b.oa21cd
        WHERE home_activity = 'Y'
        AND o_lat IS NOT NULL and o_lon IS NOT NULL
    )

    SELECT
        t.userid, home_lsoa, home_lon, home_lat,
        SUM(CASE
                WHEN t.home_activity = 'N' AND t.activity_type != 'Work' AND
                    SQRT(POW(t.o_lon - h.home_lon,2)+POW(t.o_lat - h.home_lat,2)) <= {rg_med}
                THEN 1 ELSE 0
            END) * 1.0 / NULLIF(SUM(CASE WHEN t.home_activity = 'N' THEN 1 ELSE 0 END), 0) AS cont_nonhome
    FROM (SELECT * FROM {df_path} where o_lat is not null and o_lon is not null) t
    JOIN home_tab h ON t.userid = h.userid
    GROUP BY t.userid, home_lsoa, home_lon, home_lat
"""

self_df = con.execute(self_cont).df()


####Data exploration

Let's explore what the distribution of self-containment looks like across our sample users.

First, take a look at the table we've just created:
- **userid**: unique user id
- **home_lat**: latitude of user's home location
- **home_lon**: longitude of user's home location
- **home_lsoa**: Lower Super Output Area (LSOA) containing the point (**home_lat**,**home_lon**)
- **cont_nonhome**: self-containment, across all non-home activities

As you can see in the histogram below, the distribution of self-containment is wide and symmetric. The average user spends about 45% of their time close to home (within the median radius of gyration), but 10.3% spend more than 80% of their time close to home and 17.9% spend less than 20% of their time close to home.


In [ ]:
plt.hist(self_df.cont_nonhome,bins=60,color='lightgrey',edgecolor='grey',alpha=.5)
plt.axvline(self_df[self_df.userid==sample_user].cont_nonhome.values[0],
            color='darkblue',
            linewidth=1.5,
            label='Sample user 1')
plt.axvline(self_df[self_df.userid==sample_user2].cont_nonhome.values[0],
            color='chocolate',
            linewidth=1.5,
            label='Sample user 2')
plt.axvline(self_df.cont_nonhome.mean(),
            color='dimgrey',
            linewidth=2,
            linestyle='-.',
            label=f'Mean: {np.round(self_df.cont_nonhome.mean(),3)*100}%')
plt.axvline(self_df.cont_nonhome.median(),
            color='dimgrey',
            linewidth=2,
            linestyle='dotted',
            label=f'Median: {np.round(self_df.cont_nonhome.median(),3)*100}%')
plt.legend()
plt.title('Self-containment (%)')


####**Exercise** (optional)

Measures of self-containment have been used to measure "local living" as advocated for by the [15-minute city](https://en.wikipedia.org/wiki/15-minute_city) planning paradigm --- the idea that people should live within a 15-minute walk of their basic needs.

We can walk about 1.5 km in 15 minutes. Can you use this information to modify our self-containment measure into a rough measure of "15-minute city living"?  What types of mobility patterns within our data seem to be associated with "15-minute" living patterns?

In [ ]:
## First, we'll need to change the value we're comparing against from the median radius of gyration.
# rg_15 = # change this value

# def self_containment_15(user_df): # redefine the function
#     user_df = user_df.copy()
#     if (len(user_df[user_df.activity_type=='Home'])>0) & (len(user_df[user_df.activity_type!='Home'])>0):
#       home_pt = Point(*user_df[user_df.activity_type=='Home'][['o_lon','o_lat']].values[0])
#       user_df['dist_from_home'] = user_df['geometry'].apply(lambda x: x.distance(home_pt))
#       all_nonhome_activities = user_df[user_df.activity_type!='Home']
#       close_nonhome_activities = all_nonhome_activities[all_nonhome_activities.dist_from_home<rg_15]
#       return (len(close_nonhome_activities)/len(all_nonhome_activities))


# self_df_15 = travel_data.groupby('userid').progress_apply(self_containment).reset_index(name='cont_nonhome')

## Now you have a table of the percentage of users' activities which are roughly a 15-minute walk from home.
## Identify some users with high "15-minute" living as in the previous exercise and plot their data below!

### Social interaction potential

Finally, we are going to calculate social interaction potential --- a measure not just of individuals' mobility patterns but of **potential exposure between individuals as they move throughout the city**.

Social interaction potential is a measure of co-location with other people:  
$$SIP_{i,\text{ }total} = \sum_{k\in{stays_i}} \sum_{j\in k} Dur^{i,j}*(\epsilon - Dis^{i,j})$$
$$SIP_{i,\text{ }self} = \sum_{k\in{stays_i}} \sum_{j\in k: q_i = q_j} Dur^{i,j}*(\epsilon - Dis^{i,j})$$
$$SIP_{i,\text{ }pct} = \frac{SIP_{i,\text{ }self}}{SIP_{i,\text{ }total}}$$
Here,  $k \in stays_i$ are all locations that individual $i$ visits, $j$ are all individuals who also visit $k$, $Dur^{i,j}$ is the overlap of duration between user $i$ and user $j$ in location $k$, and $Dis^{i,j}$ is the average distance between user $i$ and user $j$ while visiting location $k$. $\epsilon$ is a distance threshold chosen to represent meaningful encounters. $q_i$ is the income quintile of user $i$ --- $S_{i,self}$ represents social interaction potential of user $i$ with members of its own income group.

**If an individual has high social interaction potential, they are more frequently in close proximity to other people and thus have a larger possibility of social interaction with others.** We're going to use the idea of social interaction potential to create three measures:  
- **sip_tot**: total social interaction potential with other users in the data
- **sip_self**: total social interaction potential with other users in the data of your same income decile
- **sip_pct**: **sip_self**/**sip_tot**, or percentage of total social interaction that is with members of your own income group (a measure of income segregation)

Because this is a measure of co-location between any one user and all other users, it inherently involves looking at the full dataset at once. Thus, we are going to go ahead and calculate social interaction potential for our full set of users instead of just our two sample users.

**First, we need to identify overlaps between users** --- instances where two users are in the same place at the same time. In order to approximate "same place," we use h3 hexagons at resolution 8 --- hexagons with approximately 500 meter edges. We identify all instances where two users are in the same hexagon at the same time and calculate the duration of their overlap. Our measures of social interaction potential will add up the duration of all co-exposures across the dataset.

To illustrate, we plot below all users in h3 hex 88194ad16dfffff, on November 26 at 5PM, colored by their income quintile. To select a random hex code / day / hour triplet containing overlaps, uncomment the second chunk of lines below.

In [ ]:
## Choose a...
hx = '88194ad16dfffff' # Hex code
d = 26 # Day of the month
hr = 17 # Hour of the day

# # Uncomment the lines below to select a new hex code / day / hour triplet
# gr_df = con.execute(f"""select COUNT(*) AS num_overlaps, o_h8, EXTRACT(DAY FROM start_time) AS day, EXTRACT(HOUR FROM start_time) AS hour from {df} GROUP BY EXTRACT(DAY FROM start_time), EXTRACT(HOUR FROM start_time), o_h8""").df()
# rand = gr_df[gr_df['num_overlaps'] >= 10].sample(1) # 88194ad16dfffff: November 26, 17:00
# hx = rand['o_h8'].values[0]
# d = rand['day'].values[0]
# hr = rand['hour'].values[0]

overmap = plot_overlaps(h3_code=hx,day=d,hour=hr)
overmap


Now, the first step is identifying all co-locations in our dataset---pairs of users who are in the same place at the same time, like every pair of users illustrated in the map above. We will create a table which contains the following columns:
- **user1**: unique id of user1
- **user2**: unique id of user2
- **overlap_start**: max(start_time$_{user1}$, start_time$_{user2}$); the time at which the second user arrives
- **overlap_end**: min(start_time$_{user1}$, start_time$_{user2}$); the time at which the first user leaves
- **hex**: hex3 geohash at resolution level 8
- **distance_meters**: distance between the two users during the activity (we can filter on this value if we want to only focus on users who are quite close to one another)
- **overlap_duration**: duration of overlap, equivalent to the difference **overlap_end** - **overlap_start**

Again, we're going to first demonstrate how to do this in Python and then implement table in SQL, because it is very computationally intensive.

####Step 1: Identify overlaps between users.


In [ ]:
# Read in income data: median income of each Middle Super Output Area (MSOA).
inc = pd.read_csv(f'{input_path}msoa_income.csv')
Q = 10 # Number of income quintiles to divide the data into
inc['inc_q'] = np.ceil(inc['inc'].rank(pct=True)*Q)

# Add a column representing the start date; we will use this for our initial set
# of potential overlaps.
travel_data['date'] = travel_data.start_time.dt.date

# Join our travel_data table to itself on the columns o_h8 and date
# This creates a record for each pair of users who appear on the same day,
# in the same h3 geohash resolution 8 and the same time.
overlaps = travel_data.merge(travel_data,left_on=['o_h8','date'],
                             right_on=['o_h8','date'],
                             suffixes = ['_a','_b'])
overlaps = overlaps[overlaps.userid_a!=overlaps.userid_b] # Avoid self-overlaps.

# Calculate duration of overlap ---
# the time between when the last of the two users arrives
# and the first of the two users leaves.
overlap_end = np.min([overlaps.end_time_a,overlaps.end_time_b],axis=0)
overlap_start = np.max([overlaps.start_time_a,overlaps.start_time_b],axis=0)
overlaps['overlap_duration'] = overlap_end - overlap_start
# Only retain users who overlap in time.
overlaps = overlaps[overlaps.overlap_duration > timedelta(0)]

# Calculate distance between two users during their overlap.
# (We can filter on this if we only want users who, eg, come within 50 meters)
overlaps['distance_meters'] = overlaps.progress_apply(lambda row: row.geometry_a.distance(row.geometry_b), axis=1)

# Add income data (user a)
overlaps = overlaps.merge(inc[['msoa21cd','inc']].rename(columns={'inc':'inc_a','msoa21cd':'home_msoa_a'}))
# Add income data (user b)
overlaps = overlaps.merge(inc[['msoa21cd','inc']].rename(columns={'inc':'inc_b','msoa21cd':'home_msoa_b'}))

In [ ]:
overlaps.head()

####Step 2: Calculate social interaction potential.


In [ ]:
overlaps['contact'] = overlaps.apply(lambda row: row.overlap_duration.total_seconds()/60*(1000-row.distance_meters), axis=1)
s_all = overlaps.groupby(['userid_a','home_msoa_a']).agg(s_tot=('contact', 'sum')).reset_index()
s_self = overlaps[overlaps.inc_a==overlaps.inc_b].groupby(['userid_a','home_msoa_a']).agg(s_self=('contact', 'sum')).reset_index()
social_interaction_potential = s_all.merge(s_self,on=['userid_a','home_msoa_a'])
social_interaction_potential['s_pct'] = social_interaction_potential.s_self/social_interaction_potential.s_tot


In [ ]:
social_interaction_potential.head()

####Step 1 (SQL implementation)
**Note**: Again, this could start to add up for much larger set of users --- say, 100,000 users --- or longer timeframes --- say, a full year of data. In order to run fast, efficient calculations on a larger dataset, we can use SQL. See hidden cells below for a SQL implementation of radius of gyration using the Python package DuckDB.


In [ ]:
resolution = 8
con.execute(f"""
            DROP TABLE IF EXISTS sampledata_overlaps_sip_{resolution}""")

con.execute(f"""
            CREATE TABLE IF NOT EXISTS sampledata_overlaps_sip_{resolution}(
                user1 varchar,
                user2 varchar,
                overlap_start timestamp,
                overlap_end timestamp,
                hex varchar,
                distance_meters real,
                overlap_duration real)""")

home_area = 'lsoa21cd'

df = f"read_csv('{input_path}df_samp.csv',header=1)"
overlap_q = f"""
            INSERT INTO sampledata_overlaps_sip_{resolution}
            WITH activity_data AS(
                SELECT * FROM {df} a
                INNER JOIN read_csv('{input_path}hexs.csv',header=1) hexs
                    ON a.o_h9 = hexs.user_stop_hex_9
                WHERE duration/60 >= 10
            ),

            colocations AS (
                SELECT
                    a.userid AS user1,
                    b.userid AS user2,
                    a.o_h{resolution} AS hex1,
                    b.o_h{resolution} AS hex2,
                    SQRT(POW(a.o_lon - b.o_lon,2)+POW(a.o_lat - b.o_lat,2)) AS distance_meters,
                    GREATEST(a.start_time, b.start_time) AS overlap_start,
                    LEAST(a.end_time, b.end_time) AS overlap_end
                FROM activity_data a
                JOIN activity_data b
                    ON a.userid < b.userid  -- Avoid self-matching or duplicate counts
                    AND a.start_time < b.end_time -- Identify overlaps --- same time
                    AND b.start_time < a.end_time
                    AND a.o_h{resolution} = b.o_h{resolution} -- Identify overlaps --- same place
            )

            SELECT
                user1,
                user2,
                overlap_start,
                overlap_end,
                hex1,
                distance_meters,
                DATEDIFF('second', overlap_start, overlap_end) AS overlap_duration
            FROM colocations
            """
start = time.time()
con.execute(overlap_q)
print(time.time()-start)

####Step 2 (SQL implementation)


In [ ]:
df = f"read_csv('{input_path}df_samp.csv',header=1)"
df_ov = 'sampledata_overlaps_sip_8'

dem_tab = f"read_csv('{input_path}msoa_income.csv', header = 1)"
max_dist = 1000
dem_info = f'CEILING((PERCENT_RANK() OVER (ORDER BY inc))*{Q})/{Q}'
soc_int = f"""
            WITH
                home_tab AS (
                    SELECT DISTINCT userid, a.loc_msoa as home_msoa, {dem_info} as dem_info
                    FROM {df} a
                    INNER JOIN {dem_tab} b
                    ON a.loc_msoa = b.msoa21cd
                    WHERE activity_type = 'Home'
                    AND o_lat IS NOT NULL and o_lon IS NOT NULL
                )

            SELECT SUM(overlap_duration/60 * ({max_dist} - distance_meters)/1000) AS sip_tot,
                   SUM(
                    CASE
                        WHEN one.dem_info = two.dem_info THEN
                            overlap_duration/60 * ({max_dist} - distance_meters)/1000
                        ELSE 0
                    END
                ) AS sip_self,
                user1 as userid,
                one.home_msoa
            FROM {df_ov} ov
            INNER JOIN home_tab one
            ON one.userid = ov.user1
            INNER JOIN home_tab two
            ON two.userid = ov.user2
            AND distance_meters < {max_dist}
            GROUP BY user1, one.home_msoa
            """

soc_df = con.execute(soc_int).df()
soc_df['sip_pct'] = soc_df.sip_self/soc_df.sip_tot


In [ ]:
plt.hist(soc_df.sip_pct,bins=60,color='lightgrey',edgecolor='grey',alpha=.5)
try:
  plt.axvline(soc_df[soc_df.userid==sample_user].sip_pct.values[0],
            color='darkblue',
            linewidth=1.5,
            label='Sample user 1')
except IndexError:
  print('User not present --- not enough data!')

try:
  plt.axvline(soc_df[soc_df.userid==sample_user2].sip_pct.values[0],
              color='chocolate',
              linewidth=1.5,
              label='Sample user 2')
except IndexError:
  print('User not present --- not enough data!')

plt.axvline(soc_df.sip_pct.mean(),
            color='dimgrey',
            linewidth=2,
            linestyle='-.',
            label=f'Mean: {np.round(soc_df.sip_pct.mean(),3)*100}%')
plt.axvline(soc_df.sip_pct.median(),
            color='dimgrey',
            linewidth=2,
            linestyle='dotted',
            label=f'Median: {np.round(soc_df.sip_pct.median(),3)*100}%')
plt.legend()
plt.title('SIP_self (%)')

#### This looks a bit weird, right?

You may have noticed that one or both of our sample users do not even show up in this social interaction potential dataset. That's because, given the fact that we're only observing 5,000 users across all of London, we don't actually observe many overlaps in space and time.

This is also why our distribution is highly bi-modal --- most people we only observe them interacting with one other user, so it's highly likely that self-SIP = 0 (that other user is of a different income quintile) or self-SIP = 1 (that other user is of the same income quintile).

**Unfortunately, calculating social interaction potential using our small sample of 5,000 users isn't super meaningful** --- a deep understanding of social interaction potential requires a critical mass of users in order to observe a representative sample of the others that a given user is exposed to as they move throughout the city. Our sample of just 5,000 users doesn't allow us to observe many co-locations.

**We have calculated average social interaction potential for each home LSOA across our full sample of several hundred thousand users** and provided it in the tutorial files. Let's explore the data!

In [ ]:
soc_df_alldata = pd.read_csv(f'{input_path}mobility_characteristics.csv')
print(soc_df_alldata)

In [ ]:
soc_df_alldata = pd.read_csv(f'{input_path}mobility_characteristics.csv')
soc_df_alldata = bfcs_london.merge(soc_df_alldata,left_on='lsoa21cd',right_on='home_lsoa')

plt.hist(soc_df_alldata.s_pct,bins=60,color='lightgrey',edgecolor='grey',alpha=.5)

plt.axvline(soc_df_alldata.s_pct.mean(),
            color='dimgrey',
            linewidth=2,
            linestyle='-.',
            label=f'Mean: {np.round(soc_df_alldata.s_pct.mean(),3)*100}%')
plt.axvline(soc_df_alldata.s_pct.median(),
            color='dimgrey',
            linewidth=2,
            linestyle='dotted',
            label=f'Median: {np.round(soc_df_alldata.s_pct.median(),3)*100}%')
plt.legend()
plt.title('SIP_self (%)')

In the average LSOA, around 55% of residents' interactions are with members of their own income group --- 5.5x more than what might be expected if people encountered each other randomly in space! There is a lot of variation around this average.

###Social and spatial patterns

As we did with radius of gyration, **we have also calculated average values at the home LSOA level for each of our remaining two metrics** --- self-containment and social interaction potential --- across our full sample of several hundred thousand users and provided it in the tutorial files.

We also have information on median income and distance from the city center of each neighborhood.

The relevant columns are below:
- **lsoa21cd**: unique id of LSOA
- **sip_tot**: total social interaction potential
- **sip_self**: self-interaction potential (social interaction potential which is with others of the same income group)
- **sip_pct**: **sip_self**/**sip_tot**
- **cont_all**: self-containment across all activities
- **cont_nonhome**: self-containment across non-home activities
- **rad_gyr**: radius of gyration
- **pctile_50**: median income within the LSOA --- an experimental statistic documented [here](https://www.ons.gov.uk/peoplepopulationandcommunity/personalandhouseholdfinances/incomeandwealth/datasets/adminbasedincomestatisticsdataforindividualsenglandandwales)
- **dist_from_center**: distance to the center of London, represented by the geographic center of the Greater London Area (531213.287 Easting 179649.909 Northing)

Earlier in the notebook, we explored differences across space and socioeconomic groups with respect to radius of gyration earlier in the notebook. On your own, try exploring for self-containment and social interaction potential!

In [ ]:
## Load in data
soc_df_alldata = pd.read_csv(f'{input_path}mobility_characteristics.csv')
income = pd.read_csv(f'{input_path}lsoa_income.csv')
soc_df_alldata = soc_df_alldata.merge(income[['lsoa21cd','pctile_50']].rename(columns={'lsoa21cd':'home_lsoa'}),on='home_lsoa')

## Add geography
soc_df_alldata = bfcs_london.merge(soc_df_alldata.rename(columns={'home_lsoa':'lsoa21cd'}),on='lsoa21cd')

## We're going to use the geographic center of the GLA as our "city center" to help us think about spatial trends ---
## there are lots of ways to think about city centers, this isn't necessarily the best one.
london_center = soc_df_alldata.dissolve().centroid.values[0]
soc_df_alldata['dist_from_center'] = soc_df_alldata.geometry.apply(lambda x: x.centroid.distance(london_center))

soc_df_alldata.head()

#### Does self-containment exhibit spatial patterns?

Below, we have plotted $cont\_nonhome$ --- the average self-containment for residents of each LSOA.

In [ ]:
m = choropleth_folium(soc_df_alldata, 'lsoa21cd', 's_nonhome')

**The spatial trend is not as clear here.** Let's quantify this by calculating the Pearson correlation between distance to the city center and self-containment.

In [ ]:
plt.scatter(soc_df_alldata.dist_from_center, soc_df_alldata.s_nonhome, s=5)
plt.xlabel('Distance from city center')
plt.ylabel('Self-containment')

r = pearsonr(soc_df_alldata.dist_from_center, soc_df_alldata.s_nonhome)
print(f"Pearson correlation coefficient: {np.round(r[0],4)}")
print(f"p-value: {np.round(r[1],4)}")

**Here, you might have found a slight negative correlation.**

We may want to know, more generally: **is radius of gyration is clustered in space?** Let's calculate the [Moran's I statistic](https://pro.arcgis.com/en/pro-app/latest/tool-reference/spatial-statistics/h-how-spatial-autocorrelation-moran-s-i-spatial-st.htm) to evaluate how spatially correlated radius of gyration is.

In [ ]:
# Create a queen contiguity weights matrix
w = Queen.from_dataframe(soc_df_alldata)

# Alternatively: you could create a K-Nearest Neighbors contiguity matrix
# w = KNN.from_dataframe(gdf, k=5)

w.transform = 'r'  # Row-standardized weights
y = soc_df_alldata['s_nonhome'].values # Variable of interest here is radius of gyration
moran = Moran(y, w)

print("Moran's I:", moran.I)
print("p-value:", moran.p_sim)

# We can visualize this spatial autocorrelation with a moran scatterplot.
# Here, the x-axis represents radius of gyration (standard deviation-normalized)
# The y-axis is the average radius gyration of neighbors as defined by our contiguity matrix w (adjacent polygons)
fig, ax = moran_scatterplot(moran, aspect_equal=True)
plt.show()

Again, spatial clustering is **weaker but still significant** than what we saw with radius of gyration.

#### Does self-containment differ across income group?

In [ ]:
colors = ['#3066BE','#136F63','#E0CA3C','#F34213','#842F79','#3066BE','#136F63','#E0CA3C','#F34213','#842F79']
fig,axs = plt.subplots(1,2,figsize=(12.5,5))

axs[0].scatter(soc_df_alldata.pctile_50, soc_df_alldata.s_nonhome,s=5)
axs[0].set_xlabel('Median income')
axs[0].set_ylabel('Self-containment')
r = pearsonr(soc_df_alldata.pctile_50, soc_df_alldata.s_nonhome)
print(f"Pearson correlation coefficient: {np.round(r[0],4)}")
print(f"p-value: {np.round(r[1],4)}")

soc_df_alldata['quintile'] = np.ceil(soc_df_alldata.pctile_50.rank(pct=True)*5).astype(int)
sns.boxplot(x='quintile', y='rad_gyr', data=soc_df_alldata,palette=colors[:5],ax=axs[1], boxprops=dict(alpha=.8))
axs[1].set_ylabel('Self-containment')
axs[1].set_xlabel('Income quintile')


Interesting! While the spatial trend was slightly weaker, the income trend is slightly stronger for self-containment than radius of gyration: **residents of higher-income neighborhoods tend to spend less time closer to their homes**. Why might this be?

Next, on your own, you're going to repeat this social and spatial analyiss for social interaction potential!

#### **Exercise**: Repeat for our final metrics.

We have two remaining metrics to explore:

* **Total social interaction potential (sip_total)**: How many people is a given user exposed to as they move throughout their day?
* **Self-social interaction potential (sip_pct)**: How much of that social exposure is with others in one's own income group?


1. Which of these metrics vary most across space?
*   Choropleth map
*   Correlation with distance from the city centre
*  	Moran’s I


2. Which of these metrics vary most across income groups?
*  	Correlation with median income
*   Box plots across quartiles






##### **Social interaction potential (total)**



In [ ]:
#### VARIATION ACROSS SPACE

## Choropleth map

## Correlation with distance from city centre

## Moran's I

In [ ]:
#### VARIATION ACROSS INCOME

## Correlation with median income

## Box plots across quartiles

##### **Social interaction potential (self)**


In [ ]:
#### VARIATION ACROSS SPACE

## Choropleth map

## Correlation with distance from city centre

## Moran's I

In [ ]:
#### VARIATION ACROSS INCOME

## Correlation with median income

## Box plots across quartiles

#### What have you found?

Which of these metrics seems to vary most across space? Across income group?  
More generally, do the metrics seem to vary more across space or across socioeconomic group?

## Further exercises

1. **Exercise**: Among our LSOA-level mobility metrics, I included cont_all --- self-containment across all activities, including home. Can you modify the self-containment code we put together in order to calculate this metric yourself? How does self-containment change? Are the changes consistent across neighborhood, or are some neighborhoods (eg, those further from the city center) more affected?

2. **Exercise**: When calculating social interaction potential, we used resolution 8 --- considering people who appeared in the same hexagon at the same time where hexagons have roughly 500m edges. Try changing the resolution = 8
parameter in the code to recalculate with, say resolution 7 (1.5km edges) or resolution 9 (200m edges). How does social interaction potential change? Are the changes consistent across neighborhood, or are some neighborhoods (eg, those further from the city center) more affected?

3. **Discussion question**: NOMAD helps to understand how processing choices impact the calculation of mobility metrics. What processing choices, which may have happened before you received the data, might you be concerned about impacting our calculation of radius of gyration? Self-containment? Social interaction potential?